In [3]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class JaguarTrainDataset(Dataset):
    def __init__(self, csv_path, img_dir):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir

        self.labels = sorted(self.df["ground_truth"].unique())
        self.label2id = {l: i for i, l in enumerate(self.labels)}

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row.filename)
    
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"Missing file: {img_path}")
    
        image = Image.open(img_path).convert("RGB")
        label = self.label2id[row.ground_truth]
        image = self.transform(image)
        return image, label

In [4]:
import torch
import torch.nn as nn
import torchvision.models as models

class ReIDModel(nn.Module):
    def __init__(self, embedding_dim=512, num_classes=31):
        super().__init__()

        backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])

        self.embedding = nn.Linear(512, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        feat = self.backbone(x).flatten(1)
        emb = self.embedding(feat)
        emb = nn.functional.normalize(emb)
        logits = self.classifier(emb)
        return emb, logits

In [5]:
import os
import pandas as pd

df = pd.read_csv("/kaggle/input/jaguar-re-id/train.csv")

base_dir = "/kaggle/input/jaguar-re-id/train/train"

missing = []
for f in df["filename"]:
    if not os.path.exists(os.path.join(base_dir, f)):
        missing.append(f)

print("Missing files:", len(missing))

Missing files: 0


In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = JaguarTrainDataset(
    csv_path="/kaggle/input/jaguar-re-id/train.csv",
    img_dir="/kaggle/input/jaguar-re-id/train/train"
)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

model = ReIDModel(num_classes=len(dataset.labels)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
print(11)
model.load_state_dict(torch.load("reid_model.pth", map_location=device))
model.train()
for epoch in range(5):
    total_loss = 0
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        emb, logits = model(images)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: loss = {total_loss / len(loader):.4f}")

torch.save(model.state_dict(), "reid_model.pth")



Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 186MB/s]


11


FileNotFoundError: [Errno 2] No such file or directory: 'reid_model.pth'

In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms
from tqdm import tqdm


@torch.no_grad()
def extract():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )
    ])

    model = ReIDModel(num_classes=31).to(device)
    model.load_state_dict(torch.load("reid_model.pth", map_location=device))
    model.eval()

    embeddings = {}

    for i in range(1, 372):
        fname = f"test_{i:04d}.png"
        img = Image.open(os.path.join("/kaggle/input/jaguar-re-id/test/test", fname)).convert("RGB")
        img = transform(img).unsqueeze(0).to(device)

        emb, _ = model(img)
        embeddings[fname] = emb.cpu().numpy()[0]

    torch.save(embeddings, "test_embeddings.pt")

if __name__ == "__main__":
    extract()

In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

def main():
    test_df = pd.read_csv("/kaggle/input/jaguar-re-id/test.csv")
    embeddings = torch.load(
    "/kaggle/working/test_embeddings.pt",
    weights_only=False)

    sims = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        e1 = embeddings[row.query_image]
        e2 = embeddings[row.gallery_image]

        sim = cosine_similarity(
            e1.reshape(1, -1),
            e2.reshape(1, -1)
        )[0, 0]
        sim = (sim + 1) / 2

        sims.append(sim)

    submission = pd.DataFrame({
        "row_id": test_df.row_id,
        "similarity": sims
    })

    submission.to_csv("submission5.csv", index=False)

if __name__ == "__main__":
    main()

**ВАРИАНТ 2**

In [ ]:
# dataset_reid.py
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

def get_train_transforms():
    return transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )
    ])

def get_val_transforms():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )
    ])

class JaguarDataset(Dataset):
    def __init__(self, csv_path, img_dir, train=True):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir

        self.labels = sorted(self.df["ground_truth"].unique())
        self.label2id = {l: i for i, l in enumerate(self.labels)}

        self.transform = get_train_transforms() if train else get_val_transforms()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row.filename)

        image = Image.open(img_path).convert("RGB")
        label = self.label2id[row.ground_truth]

        image = self.transform(image)

        return image, label

In [ ]:
# sampler.py
import numpy as np
from torch.utils.data import WeightedRandomSampler

def create_weighted_sampler(dataset):
    labels = [dataset.label2id[x] for x in dataset.df["ground_truth"]]
    class_counts = np.bincount(labels)

    class_weights = 1. / class_counts
    sample_weights = [class_weights[l] for l in labels]

    sampler = WeightedRandomSampler(
        sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    return sampler

In [ ]:
# model_efficientnet.py
import torch
import torch.nn as nn
import torchvision.models as models

class EfficientNetReID(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()

        backbone = models.efficientnet_b3(weights="IMAGENET1K_V1")
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.embedding = nn.Linear(1536, embedding_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        emb = self.embedding(x)
        emb = nn.functional.normalize(emb, dim=1)
        return emb

In [ ]:
# backbone_swin.py
import torch
import torch.nn as nn
import torchvision.models as models

class SwinReID(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()

        backbone = models.swin_t(weights="IMAGENET1K_V1")
        self.features = backbone.features
        self.norm = backbone.norm
        self.head = nn.Identity()

        self.embedding = nn.Linear(768, embedding_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        emb = self.embedding(x)
        emb = nn.functional.normalize(emb, dim=1)
        return emb

In [ ]:
# triplet.py
import torch
import torch.nn as nn

class BatchHardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        dist = torch.cdist(embeddings, embeddings)

        loss = 0
        for i in range(len(embeddings)):
            pos_mask = labels == labels[i]
            neg_mask = labels != labels[i]

            hardest_pos = dist[i][pos_mask].max()
            hardest_neg = dist[i][neg_mask].min()

            loss += torch.relu(hardest_pos - hardest_neg + self.margin)

        return loss / len(embeddings)

In [ ]:
dataset = JaguarDataset(
    csv_path="/kaggle/input/jaguar-re-id/train.csv",
    img_dir="/kaggle/input/jaguar-re-id/train/train"
)
dataset = create_weighted_sampler(dataset)

In [ ]:
# train_efficientnet.py
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm


device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = JaguarDataset(
    csv_path="/kaggle/input/jaguar-re-id/train.csv",
    img_dir="/kaggle/input/jaguar-re-id/train/train"
)

sampler = create_weighted_sampler(dataset)

loader = DataLoader(
    dataset,
    batch_size=32,
    sampler=sampler,
    num_workers=4
)

model = EfficientNetReID().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = BatchHardTripletLoss(margin=0.3)
print(11)
for epoch in range(4):
    model.train
    total_loss = 0
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}: loss = {total_loss / len(loader):.4f}")
    torch.save(model.state_dict(), "efficientnet_reid.pth")

torch.save(model.state_dict(), "efficientnet_reid.pth")



In [ ]:
torch.save(model.state_dict(), "efficientnet_reid.pth")

In [ ]:
# extract_embeddings.py
import torch
import numpy as np
import os
from PIL import Image
from torchvision import transforms


@torch.no_grad()
def extract():
    print(111)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        )
    ])

    model = EfficientNetReID().to(device)
    model.load_state_dict(torch.load("efficientnet_reid.pth", map_location=device))
    model.eval()

    embeddings = {}

    for i in range(1, 372):
        fname = f"test_{i:04d}.png"
        img_path = f"/kaggle/input/jaguar-re-id/test/test/{fname}"
        print(img_path)

        image = Image.open(img_path).convert("RGB")
        image = transform(image).unsqueeze(0).to(device)

        emb = model(image)
        embeddings[fname] = emb.cpu().numpy()[0]

    np.save("test_embeddings.npy", embeddings)

if __name__ == "__main__":
    extract()

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    triplet_loss_fn = BatchHardTripletLoss(margin=0.3)
    ce_loss = nn.CrossEntropyLoss()

    total_loss = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        embeddings = model(images)

        loss_triplet = triplet_loss_fn(embeddings, labels)
        loss = loss_triplet

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        print(total_loss / len(loader))

    return total_loss / len(loader)

In [ ]:
# model_swin.py
import torch
import torch.nn as nn
import torchvision.models as models

class SwinReID(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()

        backbone = models.swin_t(weights="IMAGENET1K_V1")

        self.features = backbone.features
        self.norm = backbone.norm
        self.embedding = nn.Linear(768, embedding_dim)

    def forward(self, x):
        x = self.features(x)
        x = self.norm(x)
        x = x.mean(dim=1)  # global pooling

        emb = self.embedding(x)
        emb = nn.functional.normalize(emb, dim=1)

        return emb

In [ ]:
import timm

In [ ]:
class SwinReID(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            num_classes=0
        )

        self.embedding = nn.Linear(self.backbone.num_features, embed_dim)

    def forward(self, x):
        x = self.backbone(x)      # [B, C]
        x = self.embedding(x)     # [B, embed_dim]
        x = nn.functional.normalize(x, p=2, dim=1)
        return x

In [ ]:
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

#scaler = torch.amp.GradScaler('cuda')

from tqdm import tqdm
import torch.nn as nn

def train_epoch(model, loader, optimizer, device):
    model.to(device)  # <<< ВАЖНО
    model.train()

    triplet_loss_fn = BatchHardTripletLoss(margin=0.3)

    total_loss = 0.0

    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        embeddings = model(images)

        loss = triplet_loss_fn(embeddings, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pbar.set_postfix(loss=loss.item())
        #print(total_loss / len(loader))
    return total_loss / len(loader)

In [ ]:
model_swin = SwinReID()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = BatchHardTripletLoss(margin=0.3)
for _ in range(3):
    train_epoch(model_swin,loader,optimizer,'cuda')

In [ ]:
torch.save(model_swin.state_dict(), "swin_reid.pth")

In [ ]:
model_swin.save('swin_reid.pth')

In [ ]:
# extract_swin_embeddings.py
import torch
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms


class TestDataset(Dataset):
    def __init__(self, img_dir):
        self.img_dir = img_dir
        self.names = sorted(os.listdir(img_dir))

        self.transform = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        path = os.path.join(self.img_dir, name)

        img = Image.open(path).convert("RGB")
        img = self.transform(img)

        return img, name


@torch.no_grad()
def extract():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    dataset = TestDataset("/kaggle/input/jaguar-re-id/test/test")
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    model = SwinReID().to(device)
    model.load_state_dict(torch.load("swin_reid.pth", map_location=device))
    model.eval()

    embeddings = {}
    z = 0
    for images, names in loader:
        images = images.to(device)
        z += 1
        print(z)

        emb = model(images).cpu().numpy()

        for i in range(len(names)):
            embeddings[names[i]] = emb[i]

    np.save("test_embeddings.npy", embeddings)

if __name__ == "__main__":
    extract()

In [ ]:
print(len(loader))

In [ ]:
# make_submission_swin.py
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

# загрузка эмбеддингов
embeddings = np.load("test_embeddings.npy", allow_pickle=True).item()

# сортируем имена
names = sorted(embeddings.keys())

# матрица (371, D)
E = np.stack([embeddings[n] for n in names])
E = normalize(E)

# cosine similarity
S = E @ E.T

# привести к [0,1]
S = (S + 1) / 2

# загрузить test.csv
test_df = pd.read_csv("/kaggle/input/jaguar-re-id/test.csv")

name_to_idx = {n: i for i, n in enumerate(names)}

similarities = []

for _, row in test_df.iterrows():
    i = name_to_idx[row.query_image]
    j = name_to_idx[row.gallery_image]
    similarities.append(S[i, j])

submission = pd.DataFrame({
    "row_id": test_df.row_id,
    "similarity": similarities
})

submission.to_csv("submission_swin2.csv", index=False)

***ВАРИАНТ 3***

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

class DINOReID(nn.Module):
    def __init__(self, embed_dim=224):
        super().__init__()

        # DINOv2 backbone
        self.backbone = timm.create_model(
            "vit_base_patch14_dinov2.lvd142m",
            pretrained=True,
            num_classes=0,
            img_size=224  # <<< ВАЖНО
        )

        in_features = self.backbone.num_features

        self.embedding = nn.Sequential(
            nn.Linear(in_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, embed_dim)
        )

    def forward(self, x):
        x = self.backbone(x)           # [B, C]
        x = self.embedding(x)          # [B, embed_dim]
        x = F.normalize(x, p=2, dim=1) # L2 normalize
        return x

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((518, 518)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [ ]:
from tqdm import tqdm

def train_epoch(model, loader, optimizer, device):
    model.train()
    model.to(device)

    triplet_loss_fn = BatchHardTripletLoss(margin=0.3)

    total_loss = 0
    pbar = tqdm(loader)

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        embeddings = model(images)
        loss = triplet_loss_fn(embeddings, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    return total_loss / len(loader)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DINOReID(embed_dim=224).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

for epoch in range(8):
    loss = train_epoch(model, loader, optimizer, device)
    print(f"Epoch {epoch+1} | Loss: {loss:.4f}")

In [ ]:
def extract_embeddings(model, loader, device):
    model.eval()
    model.to(device)

    all_embeddings = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = images.to(device)
            embeddings = model(images)

            all_embeddings.append(embeddings.cpu())
            all_labels.append(labels)

    return torch.cat(all_embeddings), torch.cat(all_labels)

In [ ]:
def extract_test_embeddings(model, loader, device):
    model.eval()
    model.to(device)

    all_embeddings = []

    with torch.no_grad():
        for images, _ in tqdm(loader):
            images = images.to(device)
            embeddings = model(images)
            all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings)

In [ ]:
test_embeddings = extract_test_embeddings(model, loader, device)
torch.save(test_embeddings, "test_embeddings.pt")

In [ ]:
test_embeddings = extract_embeddings(model,loader,device)
torch.save(test_embeddings, "train_embeddings.pt")

In [ ]:
def extract_embeddings(model, loader, device):
    model.eval()
    model.to(device)

    all_embeddings = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader):
            images = images.to(device)
            embeddings = model(images)

            all_embeddings.append(embeddings.cpu())
            all_labels.append(labels.cpu())

    return torch.cat(all_embeddings), torch.cat(all_labels)

In [ ]:
train_embeddings, train_labels = extract_embeddings(model, loader, device)

torch.save(train_embeddings, "train_embeddings.pt")
torch.save(train_labels, "train_labels.pt")

In [ ]:
from PIL import Image
import os

class TestDataset(torch.utils.data.Dataset):

    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        img_name = self.df.iloc[idx]["gallery_image"]
        img_path = os.path.join(self.img_dir, img_name)

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        img_id = self.df.iloc[idx]["id"]

        return img, img_id

In [ ]:
test_df = pd.read_csv('/kaggle/input/jaguar-re-id/train.csv')

In [ ]:
test_dataset = TestDataset(
    test_df,
    img_dir="test_images"
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4
)

In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

model.eval()

test_df = pd.read_csv("/kaggle/input/jaguar-re-id/test.csv")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

def get_embedding(path):
    image = Image.open(path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model(image)

    emb = F.normalize(emb, dim=1)
    return emb

similarities = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):

    img1 = f"/kaggle/input/jaguar-re-id/test/test/{row.image1}"
    img2 = f"/kaggle/input/jaguar-re-id/test/test/{row.image2}"

    emb1 = get_embedding(img1)
    emb2 = get_embedding(img2)

    sim = F.cosine_similarity(emb1, emb2).item()

    # переводим [-1,1] → [0,1]
    sim = (sim + 1) / 2

    similarities.append(sim)

submission = pd.DataFrame({
    "id": test_df["id"],
    "similarity": similarities
})

submission.to_csv("submission.csv", index=False)

In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

test_df = pd.read_csv("/kaggle/input/jaguar-re-id/test.csv")

print(test_df.columns)   # посмотреть реальные имена

model.eval()

similarities = []

def get_embedding(path):
    image = Image.open(path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model(image)

    emb = F.normalize(emb, dim=1)
    return emb


for _, row in tqdm(test_df.iterrows(), total=len(test_df)):

    img1 = f"/kaggle/input/jaguar-re-id/test/test/{row['query_image']}"
    img2 = f"/kaggle/input/jaguar-re-id/test/test/{row['gallery_image']}"

    emb1 = get_embedding(img1)
    emb2 = get_embedding(img2)

    sim = F.cosine_similarity(emb1, emb2).item()

    # перевод в диапазон [0,1]
    sim = (sim + 1) / 2

    similarities.append(sim)


submission = pd.DataFrame({
    "id": test_df["id"],
    "similarity": similarities
})

submission.to_csv("submission.csv", index=False)

In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

def main():
    test_df = pd.read_csv("/kaggle/input/jaguar-re-id/test.csv")
    embeddings = torch.load(
    "/kaggle/working/test_embeddings.pt",
    weights_only=False)

    sims = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        e1 = embeddings[row.query_image]
        e2 = embeddings[row.gallery_image]

        sim = cosine_similarity(
            e1.reshape(1, -1),
            e2.reshape(1, -1)
        )[0, 0]
        sim = (sim + 1) / 2

        sims.append(sim)

    submission = pd.DataFrame({
        "row_id": test_df.row_id,
        "similarity": sims
    })

    submission.to_csv("submission7.csv", index=False)

if __name__ == "__main__":
    main()

In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm


def main():

    test_df = pd.read_csv("/kaggle/input/jaguar-re-id/test.csv")

    embeddings = torch.load(
        "/kaggle/working/test_embeddings.pt",
        weights_only=False
    )

    # все уникальные изображения
    image_names = sorted(list(set(
        test_df.query_image.tolist() +
        test_df.gallery_image.tolist()
    )))

    image_to_idx = {name: i for i, name in enumerate(image_names)}

    sims = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):

        idx1 = image_to_idx[row.query_image]
        idx2 = image_to_idx[row.gallery_image]

        e1 = embeddings[idx1]
        e2 = embeddings[idx2]

        sim = cosine_similarity(
            e1.reshape(1, -1),
            e2.reshape(1, -1)
        )[0, 0]

        sim = (sim + 1) / 2
        sims.append(sim)

    submission = pd.DataFrame({
        "row_id": test_df.row_id,
        "similarity": sims
    })

    submission.to_csv("submission7.csv", index=False)


if __name__ == "__main__":
    main()

***ВАРИАНТ 4***

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

In [ ]:
class Config:
    seed = 42
    model_name = "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k"

    img_size = 448
    embedding_dim = 1024
    num_classes = 31

    num_epochs = 10
    batch_size = 4
    grad_accum = 4

    lr = 2e-5
    weight_decay = 1e-3

    arcface_s = 30.0
    arcface_m = 0.50

    use_tta = True
    use_qe = True
    use_rerank = True

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device_type = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


seed_everything(Config.seed)

In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize((Config.img_size, Config.img_size)),
        transforms.RandomHorizontalFlip(),
        # transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.RandomAffine(degrees=15, translate=(0.15, 0.15), scale=(0.85, 1.15)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.481, 0.457, 0.408], [0.268, 0.261, 0.275]),
        transforms.RandomErasing(p=0.25),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.Resize((Config.img_size, Config.img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.481, 0.457, 0.408], [0.268, 0.261, 0.275]),
    ]
)

In [ ]:
import torch
import torch.nn.functional as F

class TripletLoss(torch.nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):

        dist = torch.cdist(embeddings, embeddings)

        loss = 0
        count = 0

        for i in range(len(labels)):

            pos_mask = labels == labels[i]
            neg_mask = labels != labels[i]

            pos_dist = dist[i][pos_mask]
            neg_dist = dist[i][neg_mask]

            hardest_pos = pos_dist.max()
            hardest_neg = neg_dist.min()

            loss += F.relu(hardest_pos - hardest_neg + self.margin)
            count += 1

        return loss / count

In [ ]:
from torch.utils.data.sampler import Sampler
import random

class PKSampler(Sampler):

    def __init__(self, labels, P=8, K=4):
        self.labels = labels
        self.P = P
        self.K = K

        self.label_to_indices = {}

        for idx, label in enumerate(labels):
            self.label_to_indices.setdefault(label, []).append(idx)

    def __iter__(self):

        labels = list(self.label_to_indices.keys())

        while True:

            selected = random.sample(labels, self.P)

            batch = []

            for label in selected:
                indices = random.sample(self.label_to_indices[label], self.K)
                batch.extend(indices)

            yield from batch

In [ ]:
import torchvision.transforms as T

train_tfms = T.Compose([
    T.RandomResizedCrop(448),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.3,0.3,0.3,0.1),
    T.RandomAffine(10),
    T.RandomGrayscale(p=0.1),
    T.RandomErasing(p=0.5),
    T.ToTensor(),
])

In [ ]:
import random

def random_resize(img):

    size = random.choice([384, 448, 512])

    return T.Resize((size, size))(img)

In [ ]:
def extract_embedding(model, img):

    sizes = [384,448,512]

    emb = []

    for s in sizes:

        img_resized = F.interpolate(img, size=(s,s))

        f1 = model(img_resized)
        f2 = model(torch.flip(img_resized, dims=[3]))

        emb.append((f1+f2)/2)

    emb = torch.stack(emb).mean(0)

    return emb

In [ ]:
models = [
    model_eva,
    model_convnext,
    model_swin
]

def ensemble_embedding(img):

    emb = []

    for m in models:

        emb.append(m(img))

    emb = torch.stack(emb).mean(0)

    return emb

In [ ]:
import os
import cv2
import timm
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from PIL import Image
from sklearn.preprocessing import normalize
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T

In [ ]:
class CFG:

    IMG_SIZE = 448
    BATCH_SIZE = 16
    EPOCHS = 10
    LR = 2e-5

    BACKBONE = "eva02_large_patch14_448"

    EMBED_DIM = 1024

    P = 8
    K = 4

    DEVICE = "cuda"

In [ ]:
import torchvision.transforms as T

train_tfms = T.Compose([

    T.RandomResizedCrop(Config.img_size),

    T.RandomHorizontalFlip(),

    T.ColorJitter(0.3,0.3,0.3),

    T.RandomAffine(10),

    T.ToTensor(),                # ОБЯЗАТЕЛЬНО ДО Normalize

    T.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    ),

    T.RandomErasing(p=0.5)

])

In [ ]:
class JaguarDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test
        if not is_test:
            unique_ids = sorted(df["ground_truth"].unique())
            self.label_map = {name: i for i, name in enumerate(unique_ids)}
            self.df["label"] = self.df["ground_truth"].map(self.label_map)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["filename"]
        img_path = self.img_dir / img_name
        try:
            img = Image.open(img_path).convert("RGB")
        except:
            img = Image.new("RGB", (Config.img_size, Config.img_size))

        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img, img_name
        return img, torch.tensor(row["label"], dtype=torch.long)

In [ ]:
class JaguarDataset(Dataset):

    def __init__(self, df, img_dir, transform=None, is_test=False):

        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test

        if not is_test:

            unique_ids = sorted(df["ground_truth"].unique())

            self.label_map = {name: i for i, name in enumerate(unique_ids)}

            self.df["label"] = self.df["ground_truth"].map(self.label_map)

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img_path = self.img_dir / row["filename"]

        try:
            img = Image.open(img_path).convert("RGB")
        except:
            img = Image.new("RGB",(Config.img_size,Config.img_size))

        if self.transform:
            img = self.transform(img)

        if self.is_test:
            return img, row["filename"]

        return img, torch.tensor(row["label"], dtype=torch.long)

In [ ]:
class PKSampler(Sampler):

    def __init__(self, labels, P, K):

        self.labels = labels
        self.P = P
        self.K = K

        self.label_dict = {}

        for i,l in enumerate(labels):
            self.label_dict.setdefault(l,[]).append(i)

        self.unique_labels = list(self.label_dict.keys())

    def __iter__(self):

        while True:

            batch = []

            P = min(self.P, len(self.unique_labels))

            selected = random.sample(self.unique_labels, P)

            for l in selected:

                idxs = self.label_dict[l]

                if len(idxs) >= self.K:
                    idx = random.sample(idxs, self.K)
                else:
                    idx = random.choices(idxs, k=self.K)

                batch.extend(idx)

            yield from batch

In [ ]:
class GeM(nn.Module):

    def __init__(self,p=3,eps=1e-6):

        super().__init__()

        self.p = nn.Parameter(torch.ones(1)*p)
        self.eps = eps

    def forward(self,x):

        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2),x.size(-1))
        ).pow(1./self.p)

In [ ]:
class ArcFace(nn.Module):

    def __init__(self,in_features,out_features,s=30,m=0.5):

        super().__init__()

        self.weight = nn.Parameter(
            torch.FloatTensor(out_features,in_features)
        )

        nn.init.xavier_uniform_(self.weight)

        self.s = s
        self.m = m

    def forward(self,x,label):

        cosine = F.linear(
            F.normalize(x),
            F.normalize(self.weight)
        )

        theta = torch.acos(
            torch.clamp(cosine,-1+1e-7,1-1e-7)
        )

        target = torch.cos(theta+self.m)

        one_hot = F.one_hot(label,num_classes=cosine.size(1))

        logits = cosine*(1-one_hot)+target*one_hot

        return logits*self.s

In [ ]:
class TripletLoss(nn.Module):

    def __init__(self,margin=0.3):

        super().__init__()
        self.margin = margin

    def forward(self,emb,label):

        dist = torch.cdist(emb,emb)

        loss = 0
        count = 0

        for i in range(len(label)):

            pos = dist[i][label==label[i]]
            neg = dist[i][label!=label[i]]

            hardest_pos = pos.max()
            hardest_neg = neg.min()

            loss += F.relu(hardest_pos-hardest_neg+self.margin)

            count+=1

        return loss/count

In [ ]:
class ReIDModel(nn.Module):

    def __init__(self,n_classes):

        super().__init__()

        self.backbone = timm.create_model(
            CFG.BACKBONE,
            pretrained=True,
            num_classes=0
        )

        self.pool = GeM()

        self.bn = nn.BatchNorm1d(CFG.EMBED_DIM)

        self.arc = ArcFace(CFG.EMBED_DIM,n_classes)

    def forward(self,x,label=None):

        feat = self.backbone.forward_features(x)

        B,N,C = feat.shape

        H=W=int(np.sqrt(N))

        feat = feat[:,1:].permute(0,2,1).reshape(B,C,H,W)

        feat = self.pool(feat).view(B,-1)

        emb = self.bn(feat)

        if label is None:
            return F.normalize(emb)

        logits = self.arc(emb,label)

        return emb,logits

In [ ]:
def train_epoch(model,loader,optimizer):

    model.train()

    ce = nn.CrossEntropyLoss()

    triplet = TripletLoss()

    total = 0

    for img,label in loader:

        img = img.cuda()
        label = label.cuda()

        emb,logits = model(img,label)

        loss1 = ce(logits,label)

        loss2 = triplet(emb,label)

        loss = loss1 + 0.5*loss2

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total += loss.item()

    return total/len(loader)

In [ ]:
def query_expansion(emb,k=5):

    sim = emb @ emb.T

    new = []

    for i in range(len(emb)):

        idx = np.argsort(-sim[i])[:k]

        new.append(emb[idx].mean(0))

    return np.stack(new)

In [ ]:
def extract_embeddings(model,loader):

    model.eval()

    embeddings = []

    with torch.no_grad():

        for img,_ in tqdm(loader):

            img = img.cuda()

            emb = []

            for s in [384,448,512]:

                img_s = F.interpolate(img,size=(s,s))

                f1 = model(img_s)

                f2 = model(torch.flip(img_s,[3]))

                emb.append((f1+f2)/2)

            emb = torch.stack(emb).mean(0)

            embeddings.append(emb.cpu())

    embeddings = torch.cat(embeddings)

    return normalize(embeddings)

In [ ]:
def compute_similarity(query,gallery):

    return query @ gallery.T

In [ ]:
def make_submission(test_df,query_emb,gallery_emb):

    sim = compute_similarity(query_emb,gallery_emb)

    preds = []

    for _,row in tqdm(test_df.iterrows()):

        q = row.query_id
        g = row.gallery_id

        score = sim[q,g]

        preds.append(score)

    sub = pd.DataFrame({

        "id":test_df.id,
        "score":preds
    })

    sub.to_csv("submission.csv",index=False)

    return sub

In [ ]:
train_df = pd.read_csv("/kaggle/input/jaguar-re-id/train.csv")

dataset = JaguarDataset(train_df,'/kaggle/input/jaguar-re-id/train/train',train_tfms)

sampler = PKSampler(
    train_df.columns.values,
    CFG.P,
    CFG.K
)

loader = DataLoader(
    dataset,
    batch_size=CFG.BATCH_SIZE,
    sampler=sampler,
    num_workers=4
)

model = ReIDModel(train_df.columns.nunique()).cuda()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LR
)

for epoch in range(CFG.EPOCHS):

    loss = train_epoch(model,loader,optimizer)

    print("epoch",epoch,"loss",loss)